# Clustering

L'obiettivo di questa esercitazione è quello di padroneggiare le principali tecniche di clustering, successiva alla fase di pre-processing preliminare, e di valutarne la performance con gli opportuni criteri di valutazione.

## Descrizione del dataset


## Pre-processing

In [1]:
import numpy as np
import pandas as pd
from sklearn._config import set_config

set_config(transform_output="pandas")

In [2]:
dd = pd.read_csv("data/dataset_esercitazione.csv")
dd

,age,sex,dzgroup,dzclass,num.co,edu,income,scoma,charges,totcst,...,crea,sod,ph,glucose,bun,urine,adlp,adls,adlsc,death
0,62.84998,male,Lung Cancer,Cancer,0,11.0,$11-$25k,0.0,9715.0,NaN,...,1.199951,141.0,7.459961,NaN,NaN,NaN,7.0,7.0,7.000000,0
1,60.33899,female,Cirrhosis,COPD/CHF/Cirrhosis,2,12.0,$11-$25k,44.0,34496.0,NaN,...,5.500000,132.0,7.250000,NaN,NaN,NaN,NaN,1.0,1.000000,1
2,52.74698,female,Cirrhosis,COPD/CHF/Cirrhosis,2,12.0,under $11k,0.0,41094.0,NaN,...,2.000000,134.0,7.459961,NaN,NaN,NaN,1.0,0.0,0.000000,1
3,42.38498,female,Lung Cancer,Cancer,2,11.0,under $11k,0.0,3075.0,NaN,...,0.799927,139.0,NaN,NaN,NaN,NaN,0.0,0.0,0.000000,1
4,79.88495,female,ARF/MOSF w/Sepsis,ARF/MOSF,1,NaN,NaN,26.0,50127.0,NaN,...,0.799927,143.0,7.509766,NaN,NaN,NaN,NaN,2.0,2.000000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9100,66.07300,male,ARF/MOSF w/Sepsis,ARF/MOSF,1,8.0,NaN,0.0,52870.0,34329.3125,...,1.099854,131.0,7.459961,188.0,21.0,NaN,NaN,0.0,0.000000,0
9101,55.15399,female,Coma,Coma,1,11.0,NaN,41.0,35377.0,23558.5000,...,5.899414,135.0,7.289062,190.0,49.0,0.0,NaN,0.0,0.000000,0
9102,70.38196,male,ARF/MOSF w/Sepsis,ARF/MOSF,1,NaN,NaN,0.0,46564.0,31409.0156,...,2.699707,139.0,7.379883,189.0,60.0,3900.0,NaN,NaN,2.525391,0
9103,47.01999,male,MOSF w/Malig,ARF/MOSF,1,13.0,NaN,0.0,58439.0,NaN,...,3.500000,135.0,7.469727,246.0,55.0,NaN,NaN,0.0,0.000000,1


In [3]:
dd.shape # 9105 obs, 43 variabili

(9105, 43)

In [4]:
dd.groupby("death").size()

death
0    2904
1    6201
dtype: int64

Dei 9105 pazienti, 6205 sono morti.

In [5]:
dd.isna().mean().sort_values(ascending = False)

adlp        0.619550
urine       0.533992
glucose     0.494234
bun         0.477979
totmcst     0.381658
alb         0.370346
income      0.327512
adls        0.314882
bili        0.285667
pafi        0.255354
ph          0.250851
prg2m       0.181109
edu         0.179462
prg6m       0.179352
totcst      0.097529
wblc        0.023284
charges     0.018891
avtisst     0.009006
crea        0.007359
race        0.004613
dnr         0.003295
dnrday      0.003295
sps         0.000110
surv2m      0.000110
scoma       0.000110
resp        0.000110
temp        0.000110
sod         0.000110
hrt         0.000110
meanbp      0.000110
aps         0.000110
surv6m      0.000110
age         0.000000
sex         0.000000
dzgroup     0.000000
num.co      0.000000
dzclass     0.000000
hday        0.000000
diabetes    0.000000
ca          0.000000
dementia    0.000000
adlsc       0.000000
death       0.000000
dtype: float64

Le variabili `adlp` e `urine` hanno più del 50% di dati mancanti. In questo caso, per semplicità, si decide di rimuoverle.

In [6]:
dd.isna().mean(axis = 1).sort_values(ascending = False)

5393    0.418605
5440    0.418605
768     0.395349
485     0.372093
3478    0.325581
          ...   
2948    0.000000
2958    0.000000
2982    0.000000
4896    0.000000
4911    0.000000
Length: 9105, dtype: float64

Nessuna osservazione ha più del 50% di variabili prive di valori registrati. Per questo motivo si sceglie di preservare tutte le righe del dataset.

In [7]:
dd = dd.drop(columns =['adlp', 'urine'])
dd.shape 

(9105, 41)

In [8]:
dd.describe()

,age,num.co,edu,scoma,charges,totcst,totmcst,avtisst,sps,aps,...,alb,bili,crea,sod,ph,glucose,bun,adls,adlsc,death
count,9105.000000,9105.000000,7471.000000,9104.000000,8.933000e+03,8217.000000,5630.000000,9023.000000,9104.000000,9104.000000,...,5733.000000,6504.000000,9038.000000,9104.000000,6821.000000,4605.000000,4753.000000,6238.000000,9105.000000,9105.000000
mean,62.650823,1.868644,11.747691,12.058546,5.999579e+04,30825.867768,28828.877838,22.610928,25.525872,37.597979,...,2.950520,2.554463,1.770961,137.568541,7.415364,159.873398,32.349463,1.637384,1.888272,0.681054
std,15.593710,1.344409,3.447743,24.636694,1.026488e+05,45780.820986,43604.261932,13.233248,9.899377,19.903852,...,0.875944,5.318448,1.686041,6.029326,0.080563,88.391541,26.792288,2.231358,2.003763,0.466094
min,18.041990,0.000000,0.000000,0.000000,1.169000e+03,0.000000,-102.719970,1.000000,0.199982,0.000000,...,0.399963,0.099991,0.099991,110.000000,6.829102,0.000000,1.000000,0.000000,0.000000,0.000000
25%,52.797000,1.000000,10.000000,0.000000,9.740000e+03,5929.566400,5177.404300,12.000000,19.000000,23.000000,...,2.399902,0.500000,0.899902,134.000000,7.379883,103.000000,14.000000,0.000000,0.000000,0.000000
50%,64.856990,2.000000,12.000000,0.000000,2.502400e+04,14452.734400,13223.500000,19.500000,23.898438,34.000000,...,2.899902,0.899902,1.199951,137.000000,7.419922,135.000000,23.000000,1.000000,1.000000,1.000000
75%,73.998960,3.000000,14.000000,9.000000,6.459800e+04,36087.937500,34223.601600,31.666656,30.199219,49.000000,...,3.599609,1.899902,1.899902,141.000000,7.469727,188.000000,42.000000,3.000000,3.000000,1.000000
max,101.847960,9.000000,31.000000,100.000000,1.435423e+06,633212.000000,710682.000000,83.000000,99.187500,143.000000,...,29.000000,63.000000,21.500000,181.000000,7.769531,1092.000000,300.000000,7.000000,7.073242,1.000000


Per evitare fenomeni di data leakage, viene richiesto di eliminare le variabili di outcome e prognosi, selezionando la variabile target. 

Se non è ancora chiaro questo concetto, lo si riprende nuovamente per renderlo chiaro. Qualsiasi task di machine learning prevede la suddivisione in training set e test set; in particolare, si deve fare in modo che le informazioni del training set e del test set non siano assolutamente condivise. 

In questo caso il data leakage, o **contaminazione dei dati**, non fa riferimento alla contaminazione tra training e test set, ma al fatto che, se l'obiettivo fosse quello di prevedere se un soggetto è morto o è vivo, **conoscere le informazioni prognostiche** significa sfruttare informazioni derivate direttamente o indirettamente dall'outcome stesso. 

Il modello deve apprendere solo da ciò che avviene prima dell'evento.

Tra le variabili che devono essere eliminate, rientrano:
- Gli score clinici `aps`, `sps`;
- Le variabili di classe diagnostica `dzgroup`, `dzclass`;

Un caso dubbio possono essere le variabili relative al costo del ricovero, in quanto è un'informazione che generalmente si apprende *durante* o *dopo* la fine del ricovero. La manterremo nel dataset. 

In [9]:
cl_to_remove = ['aps', 'sps', 'dzgroup', 'dzclass']

dd = dd.drop(columns = cl_to_remove)
dd.shape

(9105, 37)

Possiamo separare il set delle features da quello della variabile target:

In [10]:
X = dd.drop(columns = 'death')
y = dd['death']

Rispettando la consegna, si esegue lo splitting 95%-5% in training e test set:

In [11]:
from sklearn.model_selection import train_test_split

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size = 0.05, random_state = 42, stratify = y)

Adesso bisogna gestire i dati mancanti. Utilizzeremo due metodi di imputazione distinti per le variabili numeriche (la mediana) e per quelle categoriali (con un valore costante 'Unknown'); successivamente si utilizzerà l'Ordinal Encoding per le variabili categoriali.

In [12]:
from sklearn.impute import SimpleImputer

imp_num = SimpleImputer(strategy = 'median')
imp_cat = SimpleImputer(strategy = 'constant', fill_value = 'Unknown')

Naturalmente l'imputazione va fatta separatamente tra training e test set.

In [13]:
X_tr.apply(lambda x : x.unique())

age         [59.715, 64.05499, 66.50195, 80.23798, 85.7079...
sex                                            [female, male]
num.co                            [3, 1, 0, 2, 5, 4, 6, 7, 8]
edu         [8.0, 7.0, nan, 12.0, 1.0, 14.0, 11.0, 18.0, 2...
income           [nan, under $11k, $25-$50k, $11-$25k, >$50k]
scoma       [0.0, 44.0, 9.0, 41.0, 26.0, 61.0, 100.0, 37.0...
charges     [15675.0, 20604.0, 9212.0, 43960.0, 55561.0, 2...
totcst      [12458.0078, 12852.2578, nan, 22739.5156, 3194...
totmcst     [10295.7891, 15802.3125, nan, 39482.3125, 1812...
avtisst     [13.0, 47.5, 11.0, 25.0, 33.666657, 17.0, 7.5,...
race              [white, black, hispanic, nan, other, asian]
surv2m      [0.768920898, 0.238983154, 0.658935547, 0.6439...
surv6m      [0.631958008, 0.137969971, 0.354980469, 0.5429...
hday        [11, 1, 3, 6, 14, 5, 30, 22, 21, 15, 36, 12, 8...
diabetes                                               [1, 0]
dementia                                               [0, 1]
ca      

In [14]:
num_cols = X_tr.select_dtypes(include = 'number').columns
cat_cols = X_tr.select_dtypes(exclude = 'number').columns

X_tr[num_cols] = imp_num.fit_transform(X_tr[num_cols])
X_tr[cat_cols] = imp_cat.fit_transform(X_tr[cat_cols])

X_te[num_cols] = imp_num.transform(X_te[num_cols])
X_te[cat_cols] = imp_cat.transform(X_te[cat_cols])

Non applicheremo l'Ordinal Encoder indistintamente tra tutte le variabili categoriali, bensì separeremo quelle propriamente ordinali da quelle nominali, specificando l'ordine esatto delle modalità delle prime:

In [15]:
# Ordinali: income, ca
X_tr[cat_cols].apply(lambda x : x.unique())

sex                                          [female, male]
income     [Unknown, under $11k, $25-$50k, $11-$25k, >$50k]
race        [white, black, hispanic, Unknown, other, asian]
ca                                    [no, metastatic, yes]
dnr       [no dnr, dnr after sadm, dnr before sadm, Unkn...
dtype: object

In [16]:
ord_cols = ['income', 'ca']
ord_cats = [['Unknown', 'under $11k', '$11-$25k', '$25-$50k', '>$50k'], ['no', 'yes', 'metastatic']]

cat_cols = cat_cols.drop(ord_cols)
cat_cols

Index(['sex', 'race', 'dnr'], dtype='object')

In [17]:
from sklearn.preprocessing import OrdinalEncoder

enc1 = OrdinalEncoder()
enc2 = OrdinalEncoder(categories = ord_cats)

X_tr[cat_cols] = enc1.fit_transform(X_tr[cat_cols])
X_tr[ord_cols] = enc2.fit_transform(X_tr[ord_cols])


X_te[cat_cols] = enc1.transform(X_te[cat_cols])
X_te[ord_cols] = enc2.transform(X_te[ord_cols])

La fase successiva dell'analisi prevede di standardizzare le variabili (useremo z-score) e successivamente effettuare un'analisi di correlazione delle feature, al fine di identificare:
- Variabili poco correlate con la risposta;
- Variabili multicollineari (possibili ridondanze informative)

In [18]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

Possiamo verificare empiricamente che adesso le feature sono tutte a media nulla e varianza unitaria:

In [19]:
X_tr_s.describe()

,age,sex,num.co,edu,income,scoma,charges,totcst,totmcst,avtisst,...,pafi,alb,bili,crea,sod,ph,glucose,bun,adls,adlsc
count,8.649000e+03,8.649000e+03,8.649000e+03,8.649000e+03,8.649000e+03,8.649000e+03,8.649000e+03,8.649000e+03,8.649000e+03,8.649000e+03,...,8.649000e+03,8.649000e+03,8.649000e+03,8.649000e+03,8.649000e+03,8.649000e+03,8.649000e+03,8.649000e+03,8.649000e+03,8.649000e+03
mean,-2.522102e-16,-1.421250e-16,7.475938e-17,-2.160628e-16,-5.339956e-18,2.916437e-17,-1.232297e-17,-1.601987e-17,5.545339e-17,2.702839e-16,...,-2.734674e-16,-6.695483e-17,-1.396604e-17,-9.242231e-19,-1.482659e-15,-1.050082e-14,-1.816612e-16,9.201155e-17,2.608363e-17,-2.659709e-17
std,1.000058e+00,1.000058e+00,1.000058e+00,1.000058e+00,1.000058e+00,1.000058e+00,1.000058e+00,1.000058e+00,1.000058e+00,1.000058e+00,...,1.000058e+00,1.000058e+00,1.000058e+00,1.000058e+00,1.000058e+00,1.000058e+00,1.000058e+00,1.000058e+00,1.000058e+00,1.000058e+00
min,-2.858055e+00,-1.134943e+00,-1.387896e+00,-3.785347e+00,-1.043648e+00,-4.902961e-01,-5.717204e-01,-6.662429e-01,-6.483241e-01,-1.638257e+00,...,-2.356427e+00,-3.623901e+00,-4.368332e-01,-9.307752e-01,-4.560622e+00,-8.441115e+00,-2.291236e+00,-1.343004e+00,-5.603356e-01,-9.381928e-01
25%,-6.311403e-01,-1.134943e+00,-6.454471e-01,-2.554854e-01,-1.043648e+00,-4.902961e-01,-4.857159e-01,-5.167035e-01,-3.831396e-01,-8.037303e-01,...,-5.856540e-01,-3.327041e-01,-3.262404e-01,-5.137302e-01,-5.910490e-01,-2.486876e-01,-2.168294e-01,-2.940071e-01,-5.603356e-01,-9.381928e-01
50%,1.437183e-01,8.811016e-01,9.700162e-02,6.541108e-02,-2.412913e-01,-4.902961e-01,-3.384280e-01,-3.384035e-01,-2.745923e-01,-2.347347e-01,...,-1.218802e-01,-4.620172e-02,-2.598988e-01,-3.349472e-01,-9.485240e-02,4.590606e-02,-2.012323e-01,-2.440549e-01,-5.603356e-01,-4.398207e-01
75%,7.279879e-01,8.811016e-01,8.394503e-01,3.863076e-01,5.610650e-01,-1.263715e-01,4.045761e-02,7.445989e-02,-1.083266e-01,6.756584e-01,...,4.057943e-01,2.396020e-01,-1.714434e-01,8.211607e-02,5.667431e-01,4.667521e-01,-1.700383e-01,-1.941027e-01,-6.004216e-02,5.569236e-01
max,2.512711e+00,8.811016e-01,4.551694e+00,6.162444e+00,2.165778e+00,3.553311e+00,1.348712e+01,1.372673e+01,1.935018e+01,4.582762e+00,...,6.901922e+00,3.730603e+01,1.347618e+01,1.176076e+01,7.182698e+00,5.067975e+00,1.474074e+01,1.359271e+01,2.941719e+00,2.586914e+00


In [20]:
X_te_s.describe()

,age,sex,num.co,edu,income,scoma,charges,totcst,totmcst,avtisst,...,pafi,alb,bili,crea,sod,ph,glucose,bun,adls,adlsc
count,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,...,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000,456.000000
mean,0.014661,-0.003128,-0.010458,-0.020443,-0.047740,-0.054899,-0.031436,-0.038534,-0.046873,-0.016897,...,0.000088,-0.011783,0.030551,0.055569,-0.016506,-0.062751,0.057282,-0.005139,0.017854,0.057292
std,0.986490,1.001491,0.962353,1.051711,0.958604,0.918768,0.953045,0.887631,0.724771,0.993279,...,0.999992,0.901357,1.147451,1.025047,0.942031,1.038419,1.017312,0.887023,0.985058,0.970396
min,-2.803839,-1.134943,-1.387896,-3.785347,-1.043648,-0.490296,-0.561036,-0.631365,-0.645434,-1.562391,...,-2.061298,-2.765180,-0.436833,-0.990354,-2.410437,-4.415045,-1.573772,-1.143195,-0.560336,-0.938193
25%,-0.626670,-1.134943,-0.645447,-0.255485,-1.043648,-0.490296,-0.489340,-0.518299,-0.337867,-0.803730,...,-0.612005,-0.332704,-0.326240,-0.513730,-0.591049,-0.388974,-0.201232,-0.244055,-0.560336,-0.938193
50%,0.114112,0.881102,0.097002,0.065411,-0.241291,-0.490296,-0.332694,-0.338403,-0.274592,-0.196802,...,-0.121880,-0.046202,-0.259899,-0.334947,-0.094852,0.045906,-0.201232,-0.244055,-0.560336,-0.092749
75%,0.725777,0.881102,0.839450,0.065411,0.561065,-0.126371,0.006723,0.022360,-0.084051,0.514443,...,0.470848,0.097050,-0.193530,0.141759,0.566743,0.466752,-0.088154,-0.144150,-0.060042,0.558931
max,2.437795,0.881102,5.294142,3.916169,2.165778,3.553311,9.054403,7.038241,5.307035,3.596503,...,4.666058,3.674836,11.551109,5.682430,3.213125,3.482803,7.971619,5.750212,2.941719,2.550412


Adesso possiamo prima vedere se ci sono feature poco correlate alla variabile target: 

In [ ]:
soglia = 0.1

ft_to_remove = (
    X_tr_s
    .corrwith(y_tr)
    .abs()
    .sort_values(ascending = False)
    .loc[lambda x : x < soglia]
    .index
)

ft_to_remove

Index(['adls', 'num.co', 'hday', 'dementia', 'income', 'sex', 'meanbp', 'temp',
       'totmcst', 'race', 'crea', 'bili', 'glucose', 'totcst', 'bun', 'sod',
       'charges', 'diabetes', 'alb', 'ph', 'wblc', 'pafi', 'edu', 'hrt',
       'resp'],
      dtype='object')

In [28]:
X_tr_flt = X_tr_s.drop(columns = ft_to_remove)
X_tr_flt.shape

(8649, 11)

In [ ]:
soglia_mct = 0.8

coppie = (
    X_tr_flt
    .corr()
    .abs()
    .map(lambda x : x > soglia_mct)
    .unstack()
    .reset_index()
)

coppie.columns = ['var1', 'var2', 'high_corr']
coppie = coppie[coppie['high_corr'] & (coppie['var1'] < coppie['var2'])]
coppie 

,var1,var2,high_corr
37,surv2m,surv6m,True
73,prg2m,prg6m,True


L'analisi di correlazione evidenzia l'esistenza di multicollinearità tra due coppie di variabili inserite nel dataset, che essenzialmente sono la stessa ricodificata in due modi diversi. Lasceremo quella più vicina nel tempo (es. `surv2m` sopravvivenza a due mesi)

In [32]:
new_ft_to_remove = ['surv6m', 'prg6m']
ft_to_remove = ft_to_remove.union(new_ft_to_remove)

X_tr_flt = X_tr_s.drop(columns = ft_to_remove)
X_te_flt = X_te_s.drop(columns = ft_to_remove)

In [34]:
X_tr_flt.shape

(8649, 9)

In [35]:
X_te_flt.shape

(456, 9)

<div class="alert alert-block alert-info">
<b>Usare la PCA:</b> il testo consiglia di gestire la multicollinearità o con la rimozione di feature ridondanti o tramite trasformazione lineare (come la PCA). 

Se la multicollinearità riguarda solo poche coppie di variabili, spesso la rimozione manuale è la scelta più adeguata, mentre se invece sono molte le variabili tra loro correlate, allora può essere utile servirsi della PCA per ridurre la dimensionalità del dato, soprattutto se si vogliono usare modelli sensibili alla dimensionalità (KNN, SVM, reti neurali)
</div>